In [1]:
# core
import numpy as np
import pandas as pd

# COBRApy
from cobra.core.configuration import Configuration
from cobra.io import load_model
from cobra.core.solution import get_solution

#iMATpy
from imatpy.imat import imat, add_imat_constraints_
from imatpy.model_utils import read_model
from imatpy.parse_gpr import gene_to_rxn_weights
from imatpy.model_creation import generate_model

In [2]:
# set solver to glpk, this is the default optimizer installed with cobra 
# This is ridiculously slow
#alternative solver options: cplex and gurobi

Configuration().solver = "glpk"

In [4]:
# load the test salmonella model in cobrapy 
model = load_model("STM_v1_0")
model

Name,STM_v1_0
Memory address,7f35ad6fe050
Number of metabolites,1802
Number of reactions,2545
Number of genes,1271
Number of groups,0
Objective expression,1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Compartments,"cytosol, periplasm, extracellular space"


In [5]:
model.reactions[:4]

[<Reaction 3HAD180 at 0x7fc590c61490>,
 <Reaction 3HAD181 at 0x7fc590c61450>,
 <Reaction 3HAD40 at 0x7fc590c62150>,
 <Reaction 3HAD60 at 0x7fc590c63010>]

In [6]:
model.metabolites[:4]

[<Metabolite 3hoctaACP_c at 0x7fc5919567d0>,
 <Metabolite 3hcvac11eACP_c at 0x7fc591951d10>,
 <Metabolite t3c11vaceACP_c at 0x7fc591951510>,
 <Metabolite 3haACP_c at 0x7fc591bae250>]

In [7]:
model.genes[:4]

[<Gene STM0685 at 0x7fc590cce950>,
 <Gene STM1203 at 0x7fc590e78d50>,
 <Gene STM4122 at 0x7fc590ccfa90>,
 <Gene STM1473 at 0x7fc590ccfcd0>]

In [8]:
# get gene IDs from our test model 
len(model.genes)
[g.id for g in model.genes][:10]

['STM0685',
 'STM1203',
 'STM4122',
 'STM1473',
 'STM2992',
 'STM0999',
 'STM3901',
 'STM3795',
 'STM3796',
 'STM3902']

In [9]:
# sanity check the gene id by seeing which reaction a gene maps
test_gene1 = model.genes.get_by_id("STM3468")
test_gene1.reactions

frozenset({<Reaction ACOTA at 0x7fc590b78a10>,
           <Reaction SDPTA at 0x7fc58b9e21d0>})

In [10]:
# see the GPR associated with the reactions for that gene 
for r in test_gene1.reactions:
    print(r.id, "GPR:", r.gene_reaction_rule)

ACOTA GPR: STM3468 or STM1303
SDPTA GPR: STM3468


In [11]:
# A pandas series of gene expression weights (fake)
# The gene IDs need to match the actual model.gene IDs
model_weights = pd.Series({
           "STM3468": 1,
           "STM0320": 0,
           "STM0999": -1,
           "STM1473": 0,
           "STM2267": -1,
           "STM2037": 1,
           "STM4228": 0,
           "STM4227": -1,
           "STM4230": 0,
           "STM4229": 1,
       })
print(model_weights)

STM3468    1
STM0320    0
STM0999   -1
STM1473    0
STM2267   -1
STM2037    1
STM4228    0
STM4227   -1
STM4230    0
STM4229    1
dtype: int64


In [12]:
# Reaction_weights=gene_to_rxn_weights(model=model, gene_weights=model_weights)

# Reindex list of gene IDs, keep the gene weights labeled in the pandas series, and additional added genes will get weight zero
gene_weights_all = model_weights.reindex([g.id for g in model.genes], fill_value=0)

# Get the reaction weights 
reaction_weights = gene_to_rxn_weights(model=model, gene_weights=gene_weights_all)

# most reactions have a weight of zero because the gene list defined above is very small
print(reaction_weights.head())

3HAD180    0.0
3HAD181    0.0
3HAD40     0.0
3HAD60     0.0
3HAD80     0.0
dtype: float64


In [13]:
# Sanity check reaction weights using ACOTA reaction as an example
rxn_example = reaction_weights.loc["ACOTA"]
print(rxn_example)

# check why it is getting high expression
r = model.reactions.get_by_id("ACOTA")

print("GPR:", r.gene_reaction_rule)
print("Genes:", [g.id for g in r.genes])
print("Gene weights:")
gene_weights_all.loc[[g.id for g in r.genes]]

1.0
GPR: STM3468 or STM1303
Genes: ['STM3468', 'STM1303']
Gene weights:


STM3468    1
STM1303    0
dtype: int64

In [14]:
on = reaction_weights[reaction_weights == 1]
off = reaction_weights[reaction_weights == -1]
print(on)
print(off)

12PPDRtpp    1.0
12PPDStpp    1.0
ACOTA        1.0
SDPTA        1.0
dtype: float64
14GLUCANabcpp   -1.0
ACNAMtex        -1.0
MALTHXabcpp     -1.0
MALTPTabcpp     -1.0
MALTTRabcpp     -1.0
MALTTTRabcpp    -1.0
MALTabcpp       -1.0
dtype: float64


In [15]:
# run imat
imat_results = imat(
    model=model,
    rxn_weights=reaction_weights,
    epsilon=1,
    threshold=0.01
)

In [16]:
# Print the imat objective
print(f"iMAT Objective: {imat_results.objective_value}")
print("Number of active reactions:",
      (imat_results.fluxes.abs() > 1e-6).sum())

iMAT Objective: 10.0
Number of active reactions: 87


In [17]:
# Print the imat flux distribution
print(f"iMAT Flux Distribution: \n{imat_results.fluxes}")

iMAT Flux Distribution: 
3HAD180      0.000000e+00
3HAD181      0.000000e+00
3HAD40       0.000000e+00
3HAD60       0.000000e+00
3HAD80       0.000000e+00
                 ...     
12dgr2_ST    0.000000e+00
OAL_ST       0.000000e+00
OA4L_ST      0.000000e+00
OA5L_ST      5.259073e-31
OA4VL_ST     0.000000e+00
Name: fluxes, Length: 2545, dtype: float64


In [18]:
print(f"iMat Solver Status: {imat_results.status}")
print(f"Reaction Reduced Costs: \n{imat_results.reduced_costs}")
print(f"Metabolite Shadow Prices: \n{imat_results.shadow_prices}")

iMat Solver Status: optimal
Reaction Reduced Costs: 
3HAD180     NaN
3HAD181     NaN
3HAD40      NaN
3HAD60      NaN
3HAD80      NaN
             ..
12dgr2_ST   NaN
OAL_ST      NaN
OA4L_ST     NaN
OA5L_ST     NaN
OA4VL_ST    NaN
Name: reduced_costs, Length: 2545, dtype: float64
Metabolite Shadow Prices: 
3hoctaACP_c            NaN
3hcvac11eACP_c         NaN
t3c11vaceACP_c         NaN
3haACP_c               NaN
but2eACP_c             NaN
                        ..
apolpp_p               NaN
lpp_sp_p               NaN
diglyceride_prolpp_c   NaN
glyceryl_prolpp_c      NaN
prolpp_c               NaN
Name: shadow_prices, Length: 1802, dtype: float64


In [19]:
active_rxns = imat_results.fluxes[imat_results.fluxes.abs() > 1e-6]
inactive_rxns = imat_results.fluxes[imat_results.fluxes.abs() <= 1e-6]

print(active_rxns)
print(inactive_rxns)

12PPDRtex    -1.000
12PPDRtpp    -1.000
ACGK          1.000
ACGS          1.000
ACKr          0.750
              ...  
TKT2          3.875
TPI           0.125
SSALy         1.000
SUCDi        37.000
TALA          3.875
Name: fluxes, Length: 87, dtype: float64
3HAD180      0.000000e+00
3HAD181      0.000000e+00
3HAD40       0.000000e+00
3HAD60       0.000000e+00
3HAD80       0.000000e+00
                 ...     
12dgr2_ST    0.000000e+00
OAL_ST       0.000000e+00
OA4L_ST      0.000000e+00
OA5L_ST      5.259073e-31
OA4VL_ST     0.000000e+00
Name: fluxes, Length: 2458, dtype: float64


In [20]:
type(imat_results.fluxes)

pandas.core.series.Series

In [21]:
imat_results.fluxes.sort_values()[-30:]

NH4tpp            2.0000
CS                2.8750
ACONTa            2.8750
ACONTb            2.8750
TKT1              3.8750
TKT2              3.8750
TALA              3.8750
PDH               4.0000
GLCtex_copy2      5.0000
GLCptspp          5.0000
GAPD              5.1250
ENO               5.1250
ATPS4rpp          7.1400
RPE               7.7500
ATPM              8.3900
PGL              11.6250
GND              11.6250
G6PDH2r          11.6250
O2tex            18.5000
O2tpp            18.5000
EX_co2_e         20.5000
EX_h2o_e         21.5000
FRD3             34.1250
NADH18pp         34.1250
SUCDi            37.0000
CYTBO3_4pp       37.0000
THD2pp          109.9075
ASPT            131.1575
FUM             134.0325
MDH             134.9075
Name: fluxes, dtype: float64

In [22]:
imat_results.fluxes.sort_values()[-30:].to_frame()

,fluxes
NH4tpp,2.0000
CS,2.8750
ACONTa,2.8750
ACONTb,2.8750
TKT1,3.8750
TKT2,3.8750
TALA,3.8750
PDH,4.0000
GLCtex_copy2,5.0000
GLCptspp,5.0000


In [25]:
imat_results.fluxes.sort_values()[-30:].to_frame().index.to_list()

['NH4tpp',
 'CS',
 'ACONTa',
 'ACONTb',
 'TKT1',
 'TKT2',
 'TALA',
 'PDH',
 'GLCtex_copy2',
 'GLCptspp',
 'GAPD',
 'ENO',
 'ATPS4rpp',
 'RPE',
 'ATPM',
 'PGL',
 'GND',
 'G6PDH2r',
 'O2tex',
 'O2tpp',
 'EX_co2_e',
 'EX_h2o_e',
 'FRD3',
 'NADH18pp',
 'SUCDi',
 'CYTBO3_4pp',
 'THD2pp',
 'ASPT',
 'FUM',
 'MDH']

## Check that the gene id's look the same as the gene id's in the featurecounts table 

In [24]:
=model.genes[:10]

[<Gene STM0685 at 0x7fc590cce950>,
 <Gene STM1203 at 0x7fc590e78d50>,
 <Gene STM4122 at 0x7fc590ccfa90>,
 <Gene STM1473 at 0x7fc590ccfcd0>,
 <Gene STM2992 at 0x7fc590ccff10>,
 <Gene STM0999 at 0x7fc590ce4190>,
 <Gene STM3901 at 0x7fc590ce4410>,
 <Gene STM3795 at 0x7fc590ce4750>,
 <Gene STM3796 at 0x7fc590ce4a10>,
 <Gene STM3902 at 0x7fc590ce4cd0>]

## Extract list of gene_id's